# Dream-7B: Issues 44, 45, and 47

This notebook replicates the DiffuGPT-S experiments with `Dream-org/Dream-v0-Base-7B` on an A100 GPU.

It runs:
- **Issue 44:** diffusion refinement curves, measuring when final-layer argmax predictions match the final generated tokens.
- **Issue 45:** `found_time - unmask_time` histograms, measuring whether tokens become predictable before they are unmasked.
- **Issue 47:** attention entropy over diffusion time across `1000` text sequences, with mean, median, and 5th-95th percentile bands, plus token-class/position mismatch diagnostics.

The default settings target an A100. For a smoke test, lower `NUM_SEQUENCES` and `ISSUE47_NUM_SEQUENCES` first.

## 1. Install dependencies

Run this in a fresh A100 Colab/Jupyter runtime. If you already installed the project dependencies, you can skip this cell.

In [ ]:
!pip -q install --upgrade "transformers==4.51.3" "accelerate" "safetensors" "huggingface_hub" "pandas==2.2.2" "matplotlib" "tqdm"


## 2. Imports and configuration

In [ ]:
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "Dream-org/Dream-v0-Base-7B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

# A100-oriented defaults. Lower these for a quick smoke test.
NUM_SEQUENCES = 1000
ISSUE47_NUM_SEQUENCES = 1000
STEPS = 64
MAX_NEW_TOKENS = 96
ISSUE47_STEPS = 64
ISSUE47_MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0
TOP_P = 1.0
ALG = "origin"
ALG_TEMP = 0.0
DO_SAMPLE = False
SEED = 42
LARGE_LEAD_THRESHOLD = 8

# Issue 47 target attention probe.
ISSUE47_LAYER = 0
ISSUE47_HEAD = 0
ISSUE47_TARGET_TOKEN_OFFSET = 0

OUT_DIR = Path("dream7b_issue_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Load Dream-7B

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Compatibility patch for Dream remote code with some Transformers builds.
# Dream uses rope_type="default"; older/newer Transformers variants may expose
# the same implementation under "rope" instead of "default".
try:
    import transformers
    from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS
    print("transformers:", transformers.__version__)
    if "default" not in ROPE_INIT_FUNCTIONS and "rope" in ROPE_INIT_FUNCTIONS:
        ROPE_INIT_FUNCTIONS["default"] = ROPE_INIT_FUNCTIONS["rope"]
    print("ROPE_INIT_FUNCTIONS keys:", sorted(ROPE_INIT_FUNCTIONS.keys()))
except Exception as exc:
    print("ROPE compatibility patch skipped:", exc)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    device_map="auto",
).eval()

if tokenizer.mask_token_id is None:
    print("tokenizer.mask_token_id is None; mask id will be inferred from history step 0.")
else:
    print("mask_token_id:", tokenizer.mask_token_id, tokenizer.decode([tokenizer.mask_token_id]))

## 4. Prompt generation

The notebook creates 1000 mixed reasoning/creative prompts by default. The `task` field is used for stratified plots and summaries.

In [ ]:
PROMPT_TEMPLATES = [
    ("reasoning", "Question: {a} plus {b} equals what? Explain briefly.\nAnswer:"),
    ("reasoning", "Question: If a train goes {a} miles in {b} hours, what is its average speed? Explain briefly.\nAnswer:"),
    ("reasoning", "Question: A box has {a} red marbles and {b} blue marbles. How many marbles are there?\nAnswer:"),
    ("creative", "Write a vivid one-paragraph scene about {thing} after rain:\n"),
    ("creative", "Continue this story: The {thing} opened only when the moonlight touched\n"),
    ("creative", "Describe a strange little {thing} in a whimsical style:\n"),
]
THINGS = ["city", "library", "garden", "station", "market", "harbor", "museum", "lantern", "clocktower", "courtyard"]


def build_prompts(n):
    rows = []
    for i in range(int(n)):
        task, tmpl = PROMPT_TEMPLATES[i % len(PROMPT_TEMPLATES)]
        a = 10 + (i * 7) % 90
        b = 2 + (i * 11) % 30
        thing = THINGS[i % len(THINGS)]
        rows.append({
            "id": f"dream7b_{i:04d}",
            "task": task,
            "prompt": tmpl.format(a=a, b=b, thing=thing),
        })
    return rows

PROMPTS = build_prompts(NUM_SEQUENCES)
ISSUE47_PROMPTS = build_prompts(ISSUE47_NUM_SEQUENCES)
pd.DataFrame(PROMPTS).head()

## 5. Dream history helpers

In [ ]:
def model_device():
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device(DEVICE)


def prompt_inputs(prompt):
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        enc = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            return_dict=True,
            add_generation_prompt=True,
        )
    else:
        enc = tokenizer(prompt, return_tensors="pt")
    return {k: v.to(model_device()) for k, v in enc.items()}


def extract_history(out):
    if hasattr(out, "history"):
        return out.history
    if hasattr(out, "sequences_history"):
        return out.sequences_history
    if isinstance(out, dict) and "history" in out:
        return out["history"]
    return None


def normalize_step_seqs(hist):
    step_seqs = []
    if hist is None:
        return step_seqs
    if isinstance(hist, (list, tuple)):
        for s in hist:
            arr = s.detach().cpu() if isinstance(s, torch.Tensor) else torch.tensor(s)
            if arr.dim() == 2:
                step_seqs.append(arr[0].tolist())
            elif arr.dim() == 1:
                step_seqs.append(arr.tolist())
            else:
                raise ValueError(f"Unsupported history item shape: {tuple(arr.shape)}")
        return step_seqs
    if isinstance(hist, torch.Tensor):
        arr = hist.detach().cpu()
        if arr.dim() == 3:
            return [arr[t, 0].tolist() for t in range(arr.shape[0])]
        if arr.dim() == 2:
            return [arr[t].tolist() for t in range(arr.shape[0])]
    raise ValueError(f"Unsupported history type: {type(hist)}")


def infer_mask_id(step_seqs, input_len):
    if tokenizer.mask_token_id is not None:
        return int(tokenizer.mask_token_id)
    from collections import Counter
    gen0 = step_seqs[0][input_len:] if step_seqs else []
    if not gen0:
        return None
    return int(Counter(gen0).most_common(1)[0][0])


@torch.inference_mode()
def run_dream_history(prompt, steps=STEPS, max_new_tokens=MAX_NEW_TOKENS):
    enc = prompt_inputs(prompt)
    input_ids = enc["input_ids"]
    attention_mask = enc.get("attention_mask", torch.ones_like(input_ids)).to(model_device())
    input_len = int(input_ids.shape[1])

    out = model.diffusion_generate(
        inputs=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=int(max_new_tokens),
        steps=int(steps),
        temperature=float(TEMPERATURE),
        top_p=float(TOP_P),
        alg=str(ALG),
        alg_temp=float(ALG_TEMP),
        output_history=True,
        return_dict_in_generate=True,
        do_sample=bool(DO_SAMPLE),
    )
    step_seqs = normalize_step_seqs(extract_history(out))
    if not step_seqs:
        raise RuntimeError("No Dream history found. Check output_history support for this model/runtime.")
    gen_end_abs = min(len(step_seqs[-1]), input_len + int(max_new_tokens))
    final_ids = [int(x) for x in step_seqs[-1][:gen_end_abs]]
    final_text = tokenizer.decode(final_ids[input_len:gen_end_abs], skip_special_tokens=True)
    mask_id = infer_mask_id(step_seqs, input_len)
    return {
        "input_len": input_len,
        "gen_end_abs": gen_end_abs,
        "step_seqs": step_seqs,
        "final_ids": final_ids,
        "final_text": final_text,
        "mask_id": mask_id,
    }


def dream_attention_mask(input_ids):
    # Direct Dream replay calls have no padding. Passing None avoids Dream remote-code
    # SDPA shape issues from 2D masks like [B, L]. diffusion_generate still receives
    # the tokenizer attention mask in run_dream_history.
    return None


@torch.inference_mode()
def predict_argmax_by_step(step_seqs, gen_start, gen_end_abs, batch_steps=2):
    preds = []
    dev = model_device()
    for start in range(0, len(step_seqs), int(batch_steps)):
        chunk = step_seqs[start:start + int(batch_steps)]
        max_len = max(len(ids) for ids in chunk)
        batch_ids = torch.zeros((len(chunk), max_len), dtype=torch.long, device=dev)
        # All history sequences in a chunk have the same Dream-generated length in
        # normal runs; if not, pad ids are only used in the rare ragged fallback.
        for i, ids in enumerate(chunk):
            ids_t = torch.tensor(ids, dtype=torch.long, device=dev)
            batch_ids[i, :len(ids)] = ids_t
        out = model(input_ids=batch_ids, attention_mask=None, return_dict=True)
        logits = out.logits[:, int(gen_start):int(gen_end_abs), :]
        preds.extend(torch.argmax(logits, dim=-1).detach().cpu().tolist())
    return [[int(x) for x in row] for row in preds]

## 6. Token helpers and issue 44/45 metrics

In [ ]:
def tok_text(token_id):
    return tokenizer.decode([int(token_id)], skip_special_tokens=False).replace("\n", "\\n").replace("\t", "\\t")


def classify_token(token_id, mask_id=None):
    text = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    stripped = text.strip()
    if mask_id is not None and int(token_id) == int(mask_id):
        return "mask"
    if text in {tokenizer.bos_token or "", tokenizer.eos_token or "", tokenizer.pad_token or ""}:
        return "special"
    if text == "" or text.isspace():
        return "whitespace"
    if "\n" in text or "\r" in text:
        return "newline"
    if stripped == "":
        return "whitespace"
    if stripped.isdigit():
        return "number"
    punct_chars = ".,;:!?-()[]{}" + chr(34) + chr(39) + "`"
    if all(ch in punct_chars for ch in stripped):
        return "punctuation"
    if stripped.isalpha():
        if text.startswith(" "):
            return "word_start"
        return "word_piece"
    if any(ch.isdigit() for ch in stripped) and any(ch.isalpha() for ch in stripped):
        return "alphanumeric"
    return "mixed"


def first_found_time(argmax_history, pos_rel, final_token):
    for t, row in enumerate(argmax_history):
        if pos_rel < len(row) and int(row[pos_rel]) == int(final_token):
            return t
    return None


def first_unmask_time(step_seqs, pos_abs, final_token, mask_id):
    for t, row in enumerate(step_seqs):
        if pos_abs >= len(row):
            continue
        tok = int(row[pos_abs])
        if mask_id is None:
            if tok == int(final_token):
                return t
        elif tok != int(mask_id) and tok == int(final_token):
            return t
    return None

## 7. Run issues 44 and 45

In [ ]:
refinement_rows = []
token_delta_rows = []
prediction_rows = []
large_leads = []
summaries = []

for i, record in enumerate(tqdm(PROMPTS, desc="Dream issues 44/45")):
    torch.manual_seed(SEED + i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + i)

    hist = run_dream_history(record["prompt"], steps=STEPS, max_new_tokens=MAX_NEW_TOKENS)
    predictions = predict_argmax_by_step(hist["step_seqs"], hist["input_len"], hist["gen_end_abs"], batch_steps=2)

    gen_positions = list(range(hist["input_len"], hist["gen_end_abs"]))
    gen_len = len(gen_positions)
    num_steps = min(len(hist["step_seqs"]), len(predictions))

    summaries.append({
        "prompt_id": record["id"],
        "task": record["task"],
        "input_len": hist["input_len"],
        "gen_end_abs": hist["gen_end_abs"],
        "num_generated_tokens": gen_len,
        "num_steps": num_steps,
        "mask_id": hist["mask_id"],
        "final_text": hist["final_text"],
    })

    for t in range(num_steps):
        pred_row = predictions[t]
        step_ids = hist["step_seqs"][t]
        correct = 0
        masked = 0
        current_gen_ids = []
        for pos_rel, pos_abs in enumerate(gen_positions):
            final_tok = int(hist["final_ids"][pos_abs])
            if pos_rel < len(pred_row) and int(pred_row[pos_rel]) == final_tok:
                correct += 1
            if hist["mask_id"] is not None and pos_abs < len(step_ids):
                masked += int(int(step_ids[pos_abs]) == int(hist["mask_id"]))
            if pos_abs < len(step_ids):
                current_gen_ids.append(int(step_ids[pos_abs]))
        refinement_rows.append({
            "prompt_id": record["id"],
            "task": record["task"],
            "step": t,
            "progress": t / max(num_steps - 1, 1),
            "num_generated_tokens": gen_len,
            "num_correct_predictions": correct,
            "correct_fraction": correct / gen_len if gen_len else 0.0,
            "num_masked_tokens": masked,
            "masked_fraction": masked / gen_len if gen_len else 0.0,
        })
        prediction_rows.append({
            "prompt_id": record["id"],
            "task": record["task"],
            "step": t,
            "progress": t / max(num_steps - 1, 1),
            "correct_fraction": correct / gen_len if gen_len else 0.0,
            "current_generated_text": tokenizer.decode(current_gen_ids, skip_special_tokens=False),
            "argmax_generated_text": tokenizer.decode([int(x) for x in pred_row], skip_special_tokens=False),
        })

    for pos_rel, pos_abs in enumerate(gen_positions):
        final_tok = int(hist["final_ids"][pos_abs])
        found = first_found_time(predictions, pos_rel, final_tok)
        unmask = first_unmask_time(hist["step_seqs"][:num_steps], pos_abs, final_tok, hist["mask_id"])
        delta = None if found is None or unmask is None else found - unmask
        lead = None if delta is None else -delta
        row = {
            "prompt_id": record["id"],
            "task": record["task"],
            "pos_abs": pos_abs,
            "pos_rel_gen": pos_rel,
            "pos_norm_gen": pos_rel / max(gen_len - 1, 1),
            "final_token_id": final_tok,
            "final_token_str": tok_text(final_tok),
            "final_class": classify_token(final_tok, hist["mask_id"]),
            "found_time": -1 if found is None else found,
            "unmask_time": -1 if unmask is None else unmask,
            "delta_found_minus_unmask": None if delta is None else delta,
            "lead_steps_unmask_minus_found": None if lead is None else lead,
        }
        token_delta_rows.append(row)
        if lead is not None and lead >= LARGE_LEAD_THRESHOLD:
            large_leads.append({**row, "prompt": record["prompt"], "final_text": hist["final_text"]})

refinement_df = pd.DataFrame(refinement_rows)
token_delta_df = pd.DataFrame(token_delta_rows)
prediction_df = pd.DataFrame(prediction_rows)
summaries_df = pd.DataFrame(summaries)
refinement_df.head(), token_delta_df.head(), summaries_df.head()

## 8. Plot issue 44: diffusion refinement

In [ ]:
plt.figure(figsize=(9, 5))
curve_df = refinement_df.groupby(["task", "step"], as_index=False).agg(
    progress=("progress", "mean"),
    correct_fraction=("correct_fraction", "mean"),
    masked_fraction=("masked_fraction", "mean"),
)
for task, sub in curve_df.groupby("task"):
    sub = sub.sort_values("progress")
    plt.plot(sub["progress"], sub["correct_fraction"], linewidth=2.2, label=f"{task}: correct argmax")
    plt.plot(sub["progress"], 1 - sub["masked_fraction"], linestyle="--", linewidth=1.8, label=f"{task}: unmasked fraction")
plt.xlabel("Diffusion progress")
plt.ylabel("Fraction")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_refinement_curve.png", dpi=200)
plt.show()

## 9. Plot issue 45: `found_time - unmask_time`

In [ ]:
work = token_delta_df.dropna(subset=["delta_found_minus_unmask"]).copy()
work["delta_found_minus_unmask"] = work["delta_found_minus_unmask"].astype(int)

plt.figure(figsize=(9, 5))
if len(work):
    bins = range(int(work["delta_found_minus_unmask"].min()), int(work["delta_found_minus_unmask"].max()) + 2)
    for task, sub in work.groupby("task"):
        plt.hist(sub["delta_found_minus_unmask"], bins=bins, alpha=0.55, label=task)
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("found_time - unmask_time")
plt.ylabel("Token count")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_delta_histogram.png", dpi=200)
plt.show()

work.groupby("task")["delta_found_minus_unmask"].describe(percentiles=[0.05, 0.5, 0.95])

## Balanced issue 45 comparison

The original issue 45 histogram uses all valid token deltas, so tasks can contribute different numbers of datapoints if they generate different token counts. This cell balances the Dream-7B comparison by downsampling each task to the same number of valid deltas, using `BALANCE_SEED = 42`.

In [ ]:
BALANCE_SEED = 42
balanced_work = token_delta_df.dropna(subset=["delta_found_minus_unmask"]).copy()
balanced_work["delta_found_minus_unmask"] = balanced_work["delta_found_minus_unmask"].astype(int)
min_task_n = int(balanced_work.groupby("task").size().min())

balanced_token_delta_df = (
    balanced_work
    .groupby("task", group_keys=False)
    .apply(lambda g: g.sample(n=min_task_n, random_state=BALANCE_SEED))
    .reset_index(drop=True)
)

balanced_counts_df = (
    balanced_token_delta_df
    .assign(
        delta_sign=lambda d: d["delta_found_minus_unmask"].map(
            lambda x: "< 0" if x < 0 else ("> 0" if x > 0 else "= 0")
        )
    )
    .groupby(["task", "delta_sign"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["< 0", "= 0", "> 0"]:
    if col not in balanced_counts_df.columns:
        balanced_counts_df[col] = 0
balanced_counts_df["n"] = balanced_counts_df[["< 0", "= 0", "> 0"]].sum(axis=1)
balanced_counts_df["pct_delta_lt_0"] = balanced_counts_df["< 0"] / balanced_counts_df["n"]

plt.figure(figsize=(9, 5))
if len(balanced_token_delta_df):
    bins = range(
        int(balanced_token_delta_df["delta_found_minus_unmask"].min()),
        int(balanced_token_delta_df["delta_found_minus_unmask"].max()) + 2,
    )
    for task, sub in balanced_token_delta_df.groupby("task"):
        plt.hist(sub["delta_found_minus_unmask"], bins=bins, alpha=0.55, label=f"{task} (n={len(sub)})")
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("found_time - unmask_time")
plt.ylabel("Token count")
plt.title(f"Dream-7B balanced issue 45 histogram, n={min_task_n} per task")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_balanced_delta_histogram.png", dpi=200)
plt.show()

balanced_counts_df

## 10. Issue 47 attention entropy helpers

In [ ]:
def get_logits_and_attentions(input_ids):
    out = model(
        input_ids=input_ids,
        attention_mask=None,
        output_attentions=True,
        return_dict=True,
    )
    if not hasattr(out, "attentions") or out.attentions is None:
        raise RuntimeError("Model did not return attentions. Check remote-code support for output_attentions=True.")
    return out.logits, out.attentions


def attention_entropy_for_target(attentions, layer, head, target_pos):
    attn = attentions[int(layer)][0, int(head), int(target_pos)].float()
    attn = torch.clamp(attn, min=0)
    p = attn / attn.sum().clamp_min(1e-12)
    entropy = -(p * torch.log(p.clamp_min(1e-12))).sum()
    norm_entropy = entropy / math.log(max(int(attn.numel()), 2))
    return float(entropy.detach().cpu()), float(norm_entropy.detach().cpu())

## 11. Run issue 47 over 1000 sequences

In [ ]:
issue47_entropy_rows = []
issue47_mismatch_rows = []

for i, record in enumerate(tqdm(ISSUE47_PROMPTS, desc="Dream issue 47")):
    torch.manual_seed(SEED + i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + i)

    hist = run_dream_history(record["prompt"], steps=ISSUE47_STEPS, max_new_tokens=ISSUE47_MAX_NEW_TOKENS)
    gen_positions = list(range(hist["input_len"], hist["gen_end_abs"]))
    gen_len = len(gen_positions)
    target_pos = min(hist["input_len"] + int(ISSUE47_TARGET_TOKEN_OFFSET), hist["gen_end_abs"] - 1)

    for step, ids in enumerate(hist["step_seqs"][:ISSUE47_STEPS]):
        input_ids = torch.tensor(ids, dtype=torch.long, device=model_device()).unsqueeze(0)
        logits, attentions = get_logits_and_attentions(input_ids)
        ent, ent_norm = attention_entropy_for_target(attentions, ISSUE47_LAYER, ISSUE47_HEAD, target_pos)
        argmax_ids = torch.argmax(logits[0], dim=-1).detach().cpu().tolist()
        progress = step / max(min(len(hist["step_seqs"]), ISSUE47_STEPS) - 1, 1)

        issue47_entropy_rows.append({
            "sequence_id": record["id"],
            "task": record["task"],
            "step": step,
            "progress": progress,
            "layer": ISSUE47_LAYER,
            "head": ISSUE47_HEAD,
            "target_pos_abs": target_pos,
            "target_pos_rel_gen": target_pos - hist["input_len"],
            "attention_entropy": ent,
            "attention_entropy_norm": ent_norm,
        })

        # Token-class/position diagnostics are collected at the final analyzed step.
        # Entropy is still tracked for every step above; this keeps the 1000-sequence
        # class analysis tractable while answering where the final mismatches are.
        if step == min(len(hist["step_seqs"]), ISSUE47_STEPS) - 1:
            for pos_abs in gen_positions:
                final_id = int(hist["final_ids"][pos_abs])
                current_id = int(ids[pos_abs]) if pos_abs < len(ids) else -1
                argmax_id = int(argmax_ids[pos_abs]) if pos_abs < len(argmax_ids) else -1
                current_class = classify_token(current_id, hist["mask_id"])
                final_class = classify_token(final_id, hist["mask_id"])
                argmax_class = classify_token(argmax_id, hist["mask_id"])
                visible_current_differs = (
                    hist["mask_id"] is not None
                    and current_id != int(hist["mask_id"])
                    and current_id != final_id
                )
                argmax_token_differs = argmax_id != final_id
                argmax_class_differs = argmax_class != final_class
                if visible_current_differs or argmax_token_differs or argmax_class_differs:
                    rel = pos_abs - hist["input_len"]
                    pos_norm = rel / max(gen_len - 1, 1)
                    issue47_mismatch_rows.append({
                        "sequence_id": record["id"],
                        "task": record["task"],
                        "step": step,
                        "progress": progress,
                        "pos_abs": pos_abs,
                        "pos_rel_gen": rel,
                        "pos_norm_gen": pos_norm,
                        "position_bin": pd.cut([pos_norm], bins=[0, .2, .4, .6, .8, 1.000001], labels=["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"], include_lowest=True)[0],
                        "current_token_id": current_id,
                        "current_token_str": tok_text(current_id) if current_id >= 0 else "",
                        "current_class": current_class,
                        "final_token_id": final_id,
                        "final_token_str": tok_text(final_id),
                        "final_class": final_class,
                        "argmax_token_id": argmax_id,
                        "argmax_token_str": tok_text(argmax_id) if argmax_id >= 0 else "",
                        "argmax_class": argmax_class,
                        "visible_current_differs_from_final": visible_current_differs,
                        "argmax_differs_from_final": argmax_token_differs,
                        "argmax_class_differs_from_final_class": argmax_class_differs,
                    })

    if (i + 1) % 50 == 0:
        pd.DataFrame(issue47_entropy_rows).to_csv(OUT_DIR / "dream7b_issue47_attention_entropy_partial.csv", index=False)
        pd.DataFrame(issue47_mismatch_rows).to_csv(OUT_DIR / "dream7b_issue47_token_mismatches_partial.csv", index=False)

issue47_entropy_df = pd.DataFrame(issue47_entropy_rows)
issue47_mismatch_df = pd.DataFrame(issue47_mismatch_rows)
issue47_entropy_df.head(), issue47_mismatch_df.head()

## 12. Plot issue 47: mean, median, 5th and 95th percentiles

In [ ]:
issue47_summary_df = issue47_entropy_df.groupby("step", as_index=False).agg(
    progress=("progress", "mean"),
    entropy_mean=("attention_entropy_norm", "mean"),
    entropy_median=("attention_entropy_norm", "median"),
    entropy_p05=("attention_entropy_norm", lambda s: s.quantile(0.05)),
    entropy_p95=("attention_entropy_norm", lambda s: s.quantile(0.95)),
    n_sequences=("sequence_id", "nunique"),
)

plt.figure(figsize=(9, 5))
plt.fill_between(issue47_summary_df["progress"], issue47_summary_df["entropy_p05"], issue47_summary_df["entropy_p95"], alpha=0.25, label="5th-95th percentile")
plt.plot(issue47_summary_df["progress"], issue47_summary_df["entropy_mean"], linewidth=2.2, label="mean")
plt.plot(issue47_summary_df["progress"], issue47_summary_df["entropy_median"], linestyle="--", linewidth=2.2, label="median")
plt.xlabel("Diffusion progress")
plt.ylabel("Normalized attention entropy")
plt.title(f"Dream-7B attention entropy, layer {ISSUE47_LAYER}, head {ISSUE47_HEAD}")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_issue47_attention_entropy_percentiles.png", dpi=200)
plt.show()

issue47_summary_df.head(), issue47_summary_df.tail()

## 13. Token class and position mismatch diagnostics

In [ ]:
if len(issue47_mismatch_df):
    issue47_class_position_df = (
        issue47_mismatch_df
        .groupby(["task", "position_bin", "final_class", "argmax_class"], observed=True)
        .agg(
            rows=("sequence_id", "size"),
            sequences=("sequence_id", "nunique"),
            mean_progress=("progress", "mean"),
            mean_pos_norm=("pos_norm_gen", "mean"),
            argmax_class_mismatch_rate=("argmax_class_differs_from_final_class", "mean"),
            argmax_token_mismatch_rate=("argmax_differs_from_final", "mean"),
            visible_current_token_mismatch_rate=("visible_current_differs_from_final", "mean"),
        )
        .reset_index()
        .sort_values(["rows", "sequences"], ascending=False)
    )
else:
    issue47_class_position_df = pd.DataFrame()

display(issue47_class_position_df.head(25))

if len(issue47_mismatch_df):
    heat = issue47_mismatch_df[issue47_mismatch_df["argmax_class_differs_from_final_class"]].copy()
    pivot = pd.pivot_table(heat, values="sequence_id", index="final_class", columns="position_bin", aggfunc="count", fill_value=0, observed=True)
    plt.figure(figsize=(10, 5))
    plt.imshow(pivot.values, aspect="auto", cmap="viridis")
    plt.colorbar(label="Mismatch rows")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("Generated position bin")
    plt.ylabel("Final token class")
    plt.title("Dream-7B argmax token-class mismatches by position")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "dream7b_issue47_token_class_position_mismatches.png", dpi=200)
    plt.show()

## Residual-stream depth probe: when and where is an unmasked token encoded?

This section asks the stronger question: **for each eventual unmasked token, at which diffusion step and transformer depth does the final token first become decodable from the residual stream before it is unmasked?**

It uses a logit-lens style probe: run Dream on a saved diffusion-history sequence with `output_hidden_states=True`, take each layer's residual stream at selected generated-token positions, optionally apply the model's final norm, project through the LM head, and check whether the final token is the top-1 decoded token.

Because this is much more expensive than issues 44/45, the defaults sample a small number of positions per sequence. Increase `RESIDUAL_NUM_SEQUENCES` and `RESIDUAL_POSITIONS_PER_SEQUENCE` on an A100 once the smoke test works.

In [ ]:
RESIDUAL_NUM_SEQUENCES = 100
RESIDUAL_POSITIONS_PER_SEQUENCE = 8
RESIDUAL_POSITION_MODE = "even"  # "even", "random", or "all"
RESIDUAL_MAX_STEPS = 64
RESIDUAL_APPLY_FINAL_NORM = True
RESIDUAL_BALANCE_BY_TASK = True
RESIDUAL_RANDOM_SEED = 42


def nested_getattr(obj, path):
    cur = obj
    for part in path.split("."):
        if not hasattr(cur, part):
            return None
        cur = getattr(cur, part)
    return cur


def get_lm_head_module():
    for path in ["lm_head", "model.lm_head", "language_model.lm_head"]:
        module = nested_getattr(model, path)
        if module is not None:
            return module
    raise RuntimeError("Could not find lm_head on Dream model.")


def get_final_norm_module():
    for path in ["model.norm", "model.model.norm", "norm", "transformer.ln_f"]:
        module = nested_getattr(model, path)
        if module is not None:
            return module
    return None


LM_HEAD = get_lm_head_module()
FINAL_NORM = get_final_norm_module()
print("LM head:", type(LM_HEAD))
print("Final norm:", type(FINAL_NORM) if FINAL_NORM is not None else None)


def choose_residual_probe_positions(gen_positions, n, mode, rng):
    gen_positions = list(gen_positions)
    if mode == "all" or len(gen_positions) <= int(n):
        return gen_positions
    if mode == "random":
        return sorted(rng.sample(gen_positions, int(n)))
    # even coverage over the generated region, including early and late positions.
    if int(n) <= 1:
        return [gen_positions[0]]
    idxs = [round(i * (len(gen_positions) - 1) / (int(n) - 1)) for i in range(int(n))]
    return [gen_positions[i] for i in sorted(set(idxs))]


@torch.inference_mode()
def layerwise_top1_for_positions(input_ids, positions):
    out = model(
        input_ids=input_ids,
        attention_mask=None,
        output_hidden_states=True,
        return_dict=True,
    )
    if out.hidden_states is None:
        raise RuntimeError("Model did not return hidden_states; output_hidden_states=True is required.")

    positions_t = torch.tensor(positions, dtype=torch.long, device=input_ids.device)
    layer_preds = []
    for layer_idx, hidden in enumerate(out.hidden_states):
        # hidden: [1, seq, d]. Select positions -> [P, d].
        h = hidden[0].index_select(0, positions_t)
        if RESIDUAL_APPLY_FINAL_NORM and FINAL_NORM is not None:
            h = FINAL_NORM(h)
        logits = LM_HEAD(h)
        preds = torch.argmax(logits, dim=-1).detach().cpu().tolist()
        layer_preds.append([int(x) for x in preds])
    return layer_preds


def residual_first_decodes_for_history(record, hist, rng):
    gen_positions = list(range(hist["input_len"], hist["gen_end_abs"]))
    positions = choose_residual_probe_positions(
        gen_positions,
        RESIDUAL_POSITIONS_PER_SEQUENCE,
        RESIDUAL_POSITION_MODE,
        rng,
    )
    final_ids_by_pos = {pos: int(hist["final_ids"][pos]) for pos in positions}
    unmask_by_pos = {
        pos: first_unmask_time(hist["step_seqs"], pos, final_ids_by_pos[pos], hist["mask_id"])
        for pos in positions
    }

    found = {
        pos: {
            "first_decode_step": None,
            "first_decode_layer": None,
            "first_pre_unmask_decode_step": None,
            "first_pre_unmask_decode_layer": None,
        }
        for pos in positions
    }
    trace_rows = []
    max_steps = min(len(hist["step_seqs"]), int(RESIDUAL_MAX_STEPS))

    for step in range(max_steps):
        ids = hist["step_seqs"][step]
        input_ids = torch.tensor(ids, dtype=torch.long, device=model_device()).unsqueeze(0)
        layer_preds = layerwise_top1_for_positions(input_ids, positions)
        progress = step / max(max_steps - 1, 1)

        for layer_idx, preds in enumerate(layer_preds):
            for j, pos in enumerate(positions):
                final_id = final_ids_by_pos[pos]
                is_correct = int(preds[j]) == int(final_id)
                if not is_correct:
                    continue
                row = found[pos]
                if row["first_decode_step"] is None:
                    row["first_decode_step"] = step
                    row["first_decode_layer"] = layer_idx
                unmask_step = unmask_by_pos[pos]
                if unmask_step is not None and step < unmask_step and row["first_pre_unmask_decode_step"] is None:
                    row["first_pre_unmask_decode_step"] = step
                    row["first_pre_unmask_decode_layer"] = layer_idx
                trace_rows.append({
                    "prompt_id": record["id"],
                    "task": record["task"],
                    "step": step,
                    "progress": progress,
                    "layer": layer_idx,
                    "pos_abs": pos,
                    "pos_rel_gen": pos - hist["input_len"],
                    "final_token_id": final_id,
                    "final_token_str": tok_text(final_id),
                    "unmask_time": -1 if unmask_by_pos[pos] is None else unmask_by_pos[pos],
                    "pre_unmask": bool(unmask_by_pos[pos] is not None and step < unmask_by_pos[pos]),
                })

    summary_rows = []
    for pos in positions:
        final_id = final_ids_by_pos[pos]
        unmask_step = unmask_by_pos[pos]
        row = found[pos]
        pre_step = row["first_pre_unmask_decode_step"]
        summary_rows.append({
            "prompt_id": record["id"],
            "task": record["task"],
            "pos_abs": pos,
            "pos_rel_gen": pos - hist["input_len"],
            "pos_norm_gen": (pos - hist["input_len"]) / max(len(gen_positions) - 1, 1),
            "final_token_id": final_id,
            "final_token_str": tok_text(final_id),
            "final_class": classify_token(final_id, hist["mask_id"]),
            "unmask_time": -1 if unmask_step is None else unmask_step,
            "first_decode_step": -1 if row["first_decode_step"] is None else row["first_decode_step"],
            "first_decode_layer": -1 if row["first_decode_layer"] is None else row["first_decode_layer"],
            "first_pre_unmask_decode_step": -1 if pre_step is None else pre_step,
            "first_pre_unmask_decode_layer": -1 if row["first_pre_unmask_decode_layer"] is None else row["first_pre_unmask_decode_layer"],
            "pre_unmask_depth_delta": "" if pre_step is None or unmask_step is None else int(pre_step - unmask_step),
            "decoded_before_unmask": bool(pre_step is not None),
        })
    return summary_rows, trace_rows

print("Residual probe defaults:", {
    "num_sequences": RESIDUAL_NUM_SEQUENCES,
    "positions_per_sequence": RESIDUAL_POSITIONS_PER_SEQUENCE,
    "position_mode": RESIDUAL_POSITION_MODE,
    "max_steps": RESIDUAL_MAX_STEPS,
    "apply_final_norm": RESIDUAL_APPLY_FINAL_NORM,
})

## Run residual-stream depth probe

This uses newly generated Dream histories for the selected prompt subset. The output table tells us, for each probed token, the earliest diffusion step and layer where the final token is top-1 decodable from the residual stream, and whether that happened before unmasking.

In [ ]:
residual_rng = random.Random(RESIDUAL_RANDOM_SEED)
if RESIDUAL_BALANCE_BY_TASK:
    by_task = {}
    for row in PROMPTS:
        by_task.setdefault(row["task"], []).append(row)
    per_task_n = max(1, RESIDUAL_NUM_SEQUENCES // max(len(by_task), 1))
    residual_prompts = []
    for task, rows in sorted(by_task.items()):
        residual_prompts.extend(rows[:per_task_n])
else:
    residual_prompts = PROMPTS[: int(RESIDUAL_NUM_SEQUENCES)]

residual_summary_rows = []
residual_trace_rows = []
for i, record in enumerate(tqdm(residual_prompts, desc="Residual depth probe")):
    torch.manual_seed(SEED + 10_000 + i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + 10_000 + i)
    hist = run_dream_history(record["prompt"], steps=STEPS, max_new_tokens=MAX_NEW_TOKENS)
    summary_rows, trace_rows = residual_first_decodes_for_history(record, hist, residual_rng)
    residual_summary_rows.extend(summary_rows)
    residual_trace_rows.extend(trace_rows)
    if (i + 1) % 10 == 0:
        pd.DataFrame(residual_summary_rows).to_csv(OUT_DIR / "dream7b_residual_depth_summary_partial.csv", index=False)
        pd.DataFrame(residual_trace_rows).to_csv(OUT_DIR / "dream7b_residual_depth_correct_events_partial.csv", index=False)

residual_depth_df = pd.DataFrame(residual_summary_rows)
residual_events_df = pd.DataFrame(residual_trace_rows)
residual_depth_df.head(), residual_events_df.head()

## Summarize residual-stream depth probe

Negative `pre_unmask_depth_delta` means the final token was top-1 decodable from some residual layer before the token was visibly unmasked. The layer histogram answers **at what depth** this first happened.

In [ ]:
residual_valid = residual_depth_df[residual_depth_df["decoded_before_unmask"]].copy()
residual_valid["pre_unmask_depth_delta"] = pd.to_numeric(residual_valid["pre_unmask_depth_delta"], errors="coerce")

residual_depth_counts_df = (
    residual_depth_df
    .groupby("task")
    .agg(
        probed_tokens=("prompt_id", "size"),
        decoded_before_unmask=("decoded_before_unmask", "sum"),
        decoded_before_unmask_rate=("decoded_before_unmask", "mean"),
        median_first_pre_unmask_step=("first_pre_unmask_decode_step", lambda s: pd.to_numeric(s, errors="coerce").replace(-1, pd.NA).dropna().median()),
        median_first_pre_unmask_layer=("first_pre_unmask_decode_layer", lambda s: pd.to_numeric(s, errors="coerce").replace(-1, pd.NA).dropna().median()),
    )
    .reset_index()
)

display(residual_depth_counts_df)

plt.figure(figsize=(9, 5))
if len(residual_valid):
    bins = range(int(residual_valid["first_pre_unmask_decode_layer"].min()), int(residual_valid["first_pre_unmask_decode_layer"].max()) + 2)
    for task, sub in residual_valid.groupby("task"):
        plt.hist(sub["first_pre_unmask_decode_layer"], bins=bins, alpha=0.55, label=f"{task} (n={len(sub)})")
plt.xlabel("First pre-unmask decodable residual layer")
plt.ylabel("Token count")
plt.title("Where final tokens first become top-1 decodable before unmasking")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_residual_first_pre_unmask_layer_histogram.png", dpi=200)
plt.show()

plt.figure(figsize=(9, 5))
if len(residual_valid):
    bins = range(int(residual_valid["pre_unmask_depth_delta"].min()), int(residual_valid["pre_unmask_depth_delta"].max()) + 2)
    for task, sub in residual_valid.groupby("task"):
        plt.hist(sub["pre_unmask_depth_delta"], bins=bins, alpha=0.55, label=f"{task} (n={len(sub)})")
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("first_pre_unmask_decode_step - unmask_time")
plt.ylabel("Token count")
plt.title("When residual stream first decodes final token before unmasking")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "dream7b_residual_pre_unmask_step_delta_histogram.png", dpi=200)
plt.show()

residual_depth_df.head()

## 14. Save outputs and download

In [ ]:
refinement_df.to_csv(OUT_DIR / "dream7b_refinement_by_step.csv", index=False)
token_delta_df.to_csv(OUT_DIR / "dream7b_token_deltas.csv", index=False)
prediction_df.to_json(OUT_DIR / "dream7b_prediction_texts.jsonl", orient="records", lines=True, force_ascii=False)
summaries_df.to_csv(OUT_DIR / "dream7b_generation_summaries.csv", index=False)
if "balanced_token_delta_df" in globals():
    balanced_token_delta_df.to_csv(OUT_DIR / "dream7b_balanced_token_deltas_seed42.csv", index=False)
if "balanced_counts_df" in globals():
    balanced_counts_df.to_csv(OUT_DIR / "dream7b_balanced_delta_counts_seed42.csv", index=False)
issue47_entropy_df.to_csv(OUT_DIR / "dream7b_issue47_attention_entropy.csv", index=False)
issue47_summary_df.to_csv(OUT_DIR / "dream7b_issue47_attention_entropy_summary.csv", index=False)
issue47_mismatch_df.to_csv(OUT_DIR / "dream7b_issue47_token_mismatches.csv", index=False)
issue47_class_position_df.to_csv(OUT_DIR / "dream7b_issue47_token_class_position_summary.csv", index=False)
if "residual_depth_df" in globals():
    residual_depth_df.to_csv(OUT_DIR / "dream7b_residual_depth_summary.csv", index=False)
if "residual_events_df" in globals():
    residual_events_df.to_csv(OUT_DIR / "dream7b_residual_depth_correct_events.csv", index=False)
if "residual_depth_counts_df" in globals():
    residual_depth_counts_df.to_csv(OUT_DIR / "dream7b_residual_depth_counts.csv", index=False)

with open(OUT_DIR / "dream7b_large_leads.json", "w", encoding="utf-8") as f:
    json.dump(sorted(large_leads, key=lambda r: r.get("lead_steps_unmask_minus_found") or 0, reverse=True), f, indent=2, ensure_ascii=False)

summary = {
    "model_name": MODEL_NAME,
    "num_sequences": NUM_SEQUENCES,
    "issue47_num_sequences": ISSUE47_NUM_SEQUENCES,
    "steps": STEPS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "issue47_steps": ISSUE47_STEPS,
    "issue47_max_new_tokens": ISSUE47_MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "alg": ALG,
    "alg_temp": ALG_TEMP,
    "do_sample": DO_SAMPLE,
    "seed": SEED,
    "issue47_layer": ISSUE47_LAYER,
    "issue47_head": ISSUE47_HEAD,
    "issue47_target_token_offset": ISSUE47_TARGET_TOKEN_OFFSET,
    "residual_num_sequences": globals().get("RESIDUAL_NUM_SEQUENCES", None),
    "residual_positions_per_sequence": globals().get("RESIDUAL_POSITIONS_PER_SEQUENCE", None),
    "residual_position_mode": globals().get("RESIDUAL_POSITION_MODE", None),
    "residual_max_steps": globals().get("RESIDUAL_MAX_STEPS", None),
    "residual_apply_final_norm": globals().get("RESIDUAL_APPLY_FINAL_NORM", None),
}
with open(OUT_DIR / "dream7b_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

!zip -qr dream7b_issue_outputs.zip dream7b_issue_outputs
print("Wrote", OUT_DIR.resolve())
print("Created dream7b_issue_outputs.zip")

In [ ]:
# Colab-only convenience download. If you are in local Jupyter, use the file browser instead.
try:
    from google.colab import files
    files.download("dream7b_issue_outputs.zip")
except Exception as exc:
    print("Download helper skipped:", exc)